# Extractive Text Summarization

In [ ]:
import pandas as pd
import numpy as np
import textwrap
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import matplotlib.pyplot as plt

In [ ]:
nltk.download("punkt")
nltk.download('punkt_tab')
nltk.download("stopwords")

In [ ]:
df = pd.read_csv("https://github.com/febse/data/raw/refs/heads/main/ta/BBC%20News%20Train.csv.zip")
df.head()

In [ ]:
doc = df.loc[df["ArticleId"] == 1833].iloc[0]

print(textwrap.fill(doc["Text"], replace_whitespace=False, fix_sentence_endings=True))

In [ ]:
from nltk.tokenize import sent_tokenize

sents = sent_tokenize(doc["Text"])
sents

In [ ]:
vectorizer = TfidfVectorizer(stop_words=stopwords.words('english'))

X = vectorizer.fit_transform(sents)
pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out()).head()

In [ ]:
X = X.toarray()
scores = np.array([np.mean(row[row != 0]) for row in X])
print("Row-wise Mean of Non-zero Elements:\n", scores)

In [ ]:
sort_idx = np.argsort(-scores)

In [ ]:
print("Top sentences:\n")

for i in sort_idx[:5]:
  print(f"%.2f: %s" % (scores[i], sents[i]))

## TextRank

In [ ]:
V_to_C = 34838 / (34838 + 6643)  # P(V->C | V)
V_to_V = 6643 / (34838 + 6643)   # P(V->C | V)

C_to_C = 31374 / (31374 + 34837) # P(C->C | C)
C_to_V = 34837 / (31374 + 34837) # P(C->V | C)

print(f"P(V->V|V): {V_to_V:.2}, P(V->C|V): {V_to_C:.2}")
print(f"P(C->C|C): {C_to_C:.2}, P(C->V|C): {C_to_V:.2}")

$$
\mathbf{p_t} = \begin{pmatrix} P(S_t = v) & P(S_t = c) \end{pmatrix} , \quad \mathbf{T} = \begin{pmatrix} P(S_{t+1} = v | S_t = v) & P(S_{t+1} = c | S_t = v) \\ P(S_{t+1} = v | S_t = c) & P(S_{t+1} = c | S_t = c) \end{pmatrix}
$$

$$
\mathbf{p_{t+1}} = \mathbf{p_t} \mathbf{T}
$$

$$
\mathbf{p^*} = \mathbf{p^*} \mathbf{T}
$$

In [ ]:
import numpy as np

p0 = np.array([0.02, 1 - 0.02])

T = np.array([[V_to_V, V_to_C],
              [C_to_V, C_to_C]])

print("Transition Matrix:\n", T)

def markov_step(p, T):
    return np.dot(p, T)

for step in range(10):
    p0 = markov_step(p0, T)
    print(f"After step {step+1}: P(V)={p0[0]:.4f}, P(C)={p0[1]:.4f}")

In [ ]:
# Compute the steady-state distribution
eigenvalues, eigenvectors = np.linalg.eig(T.T)

print("Eigenvalues:", eigenvalues)
print("Eigenvectors:\n", eigenvectors)

# Find the eigenvector corresponding to eigenvalue 1
eigenvector_to_ev_1 = eigenvectors[:, 1]

# Normalize to get probabilities
stationary_dist = eigenvector_to_ev_1 / eigenvector_to_ev_1.sum()

print(f"\nStationary distribution: {stationary_dist}")
print(f"P(V) = {stationary_dist[0]:.4f}, P(C) = {stationary_dist[1]:.4f}")

$$
\mathbf{A} \mathbf{v} = \lambda \mathbf{v}
$$

In [ ]:
# We already have th TF-IDF matrix X from before

print("X shape:", X.shape)
print(X)

In [ ]:
# Now we compute the cosine similarity matrix
# between each pair of sentences

from sklearn.metrics.pairwise import cosine_similarity

cosine_sim_matrix = cosine_similarity(X)

print("Cosine similarity shape:", cosine_sim_matrix.shape)
print("Cosine Similarity Matrix:\n", cosine_sim_matrix)

In [ ]:
# This matrix is no transition matrix, though 
# as the rows do not sum to 1.

T = cosine_sim_matrix / cosine_sim_matrix.sum(axis=1, keepdims=True)

# Check if the rows sum to one
T.sum(axis=1)

$$
P = \alpha T + (1-\alpha) E, \quad \alpha \in [0, 1]
$$

In [ ]:
alpha = 0.1

E = np.ones_like(T) / T.shape[0]
P = alpha * T + (1 - alpha) * E

In [ ]:
# Now we need to find the stationary distribution of P

evals, evecs = np.linalg.eig(P.T)
print("Eigenvalues:\n", evals)

In [ ]:
ev_stationary = evecs[:,0]
ev_stationary

In [ ]:
stationary_dist = ev_stationary / ev_stationary.sum()
stationary_dist

In [ ]:
# Rank sentences by their stationary distribution probability (TextRank)
ranked_indices = np.argsort(-stationary_dist)
print("Sentences ranked by TextRank probability:\n")
for idx in ranked_indices:
    print(f"{stationary_dist[idx]:.4f}: {sents[idx]}")